# ShellBreaker — ResNet50 Training Notebook

**Paper:** GAShellBreaker (Electronics MDPI 2025)  
**Model:** ResNet50 fine-tuned on 149×149 opcode-adjacency grayscale PNGs  
**Protocol:** 3 independent runs, 80/20 train/test split, fileless webshells held-out

## Before you start

**On your local machine**, run these commands first:
```bash
cd /home/quynh/Downloads/ShellBreaker

# 1. Regenerate clean PNGs from current dataset (if not done already)
.venv/bin/python scripts/03_build_grayscale.py

# 2. Zip everything Colab needs
zip -r output.zip \
    output/webshell_file \
    output/webshell_fileless \
    output/benign \
    output/benign_test \
    output/vocab.json \
    output/dataset.csv
```

**Upload to Google Drive** (`MyDrive/ShellBreaker/`):
- `output.zip` — PNG images + vocab + dataset CSV
- `scripts/04_train_resnet50.py` — training script

**In Colab:**
1. Runtime → Change runtime type → **T4 GPU** (or A100 if Pro)
2. Run all cells top-to-bottom
3. Expected time: **~20–40 min on T4**

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── EDIT THIS PATH to match where you uploaded your files ──
DRIVE_BASE = '/content/drive/MyDrive/ShellBreaker'

import os
print('Drive files:', os.listdir(DRIVE_BASE))

## Step 2 — Install dependencies

In [ ]:
# Colab already has torch/torchvision; install any missing packages
!pip install -q scikit-learn tqdm pillow seaborn

import torch
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA avail  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('[warn] No GPU — training will be very slow. Switch runtime to T4.')

## Step 3 — Extract dataset from Drive

In [ ]:
import os, csv

WORK_DIR = '/content/ShellBreaker'
os.makedirs(WORK_DIR, exist_ok=True)

# Extract output.zip (webshell_file/, webshell_fileless/, benign/, benign_test/, vocab.json, dataset.csv)
ZIP_PATH = f'{DRIVE_BASE}/output.zip'
!unzip -q "{ZIP_PATH}" -d "{WORK_DIR}"

# Copy the training script into WORK_DIR root (path-patching needs it here)
!cp "{DRIVE_BASE}/04_train_resnet50.py" "{WORK_DIR}/"

# ── Verify extraction ────────────────────────────────────────────
print('=== output/ structure ===')
!ls {WORK_DIR}/output/

print('\n=== PNG counts per category ===')
for cat in ['webshell_file', 'webshell_fileless', 'benign', 'benign_test']:
    d = f'{WORK_DIR}/output/{cat}'
    count = len([f for f in os.listdir(d) if f.endswith('.png')]) if os.path.isdir(d) else 0
    print(f'  {cat:<22}: {count} PNGs')

with open(f'{WORK_DIR}/output/dataset.csv') as f:
    rows = list(csv.DictReader(f))
n_ws = sum(1 for r in rows if r['type'] == 'webshell_file')
n_bn = sum(1 for r in rows if r['type'] == 'benign')
n_fl = sum(1 for r in rows if r['type'] == 'webshell_fileless')
print(f'\n=== dataset.csv ===')
print(f'  webshell_file   (train/val)  : {n_ws}')
print(f'  benign          (train/val)  : {n_bn}')
print(f'  webshell_fileless (held-out) : {n_fl}  ← excluded from training')

## Step 4 — Sanity check: visualise sample images

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import csv, random

rows = []
with open(f'{WORK_DIR}/output/dataset.csv') as f:
    for r in csv.DictReader(f):
        path = Path(r['path'])
        if not path.is_absolute():
            path = Path(WORK_DIR) / path
        rows.append((str(path), int(r['label']), r['type']))

# Sample 4 webshell + 4 benign
ws_rows = random.sample([r for r in rows if r[1] == 1], min(4, sum(1 for r in rows if r[1]==1)))
bn_rows = random.sample([r for r in rows if r[1] == 0], min(4, sum(1 for r in rows if r[1]==0)))
samples = ws_rows + bn_rows
random.shuffle(samples)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (path, label, type_) in zip(axes.flat, samples):
    ax.imshow(Image.open(path), cmap='gray')
    color = 'red' if label == 1 else 'green'
    ax.set_title(f'{"WEBSHELL" if label==1 else "benign"}\n({type_})', color=color, fontsize=8)
    ax.axis('off')
plt.suptitle('Opcode adjacency matrix PNGs (149×149)', fontsize=12)
plt.tight_layout()
plt.show()

# Show fileless samples (held-out)
fl_dir = Path(f'{WORK_DIR}/output/webshell_fileless')
fl_pngs = list(fl_dir.glob('*.png'))
if fl_pngs:
    n_show = min(4, len(fl_pngs))
    fig, axes = plt.subplots(1, n_show, figsize=(3*n_show, 3))
    if n_show == 1: axes = [axes]
    for ax, p in zip(axes, random.sample(fl_pngs, n_show)):
        ax.imshow(Image.open(p), cmap='gray')
        ax.set_title('FILELESS\n(held-out)', color='orange', fontsize=8)
        ax.axis('off')
    plt.suptitle('Fileless webshell PNGs — held-out test only', fontsize=11)
    plt.tight_layout()
    plt.show()

## Step 5 — Patch paths and run training

In [ ]:
import re

script_path = f'{WORK_DIR}/04_train_resnet50.py'
code = open(script_path).read()

# Override ROOT so all output/* paths resolve correctly inside Colab
code = re.sub(
    r'ROOT = Path\(__file__\)\.parent\.parent',
    f'ROOT = Path("{WORK_DIR}")',
    code
)
open(script_path, 'w').write(code)
print(f'ROOT patched → {WORK_DIR}')
print('Launching training...')

In [ ]:
# Run training — 20 epochs × 3 runs ≈ 20-40 min on T4
%cd {WORK_DIR}
!python 04_train_resnet50.py

## Step 6 — Inspect results

In [ ]:
import json

report = json.load(open(f'{WORK_DIR}/output/training_report.json'))

print(f"Model      : {report.get('model', 'ResNet50')}")
print(f"Epochs     : {report.get('epochs')}  |  LR: {report.get('lr')}  |  Batch: {report.get('batch_size')}")
print(f"Train pool : {report.get('train_samples')}  |  Held-out: {report.get('held_out_samples')}")

print('\n=== Per-run test results ===')
for r in report['runs']:
    t  = r['test']
    cm = t['confusion_matrix']
    print(f"  Run {r['run']}: "
          f"acc={t['accuracy']:.4f}  pre={t['precision']:.4f}  "
          f"rec={t['recall']:.4f}  f1={t['f1']:.4f}  auc={t['auc_roc']:.4f}")
    print(f"    CM: TN={cm[0][0]}  FP={cm[0][1]} | FN={cm[1][0]}  TP={cm[1][1]}")

print('\n=== 3-Run Average ===')
avg = report['average']
std = report['std']
for k in ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']:
    print(f'  {k:12s}: {avg[k]:.4f}  ±{std[k]:.4f}')
print(f"\n  Inference threshold : {report.get('inference_threshold', 0.50)}")

fl = report.get('fileless_generalisation')
if fl and fl.get('average'):
    print('\n=== Fileless generalisation (held-out — never in training) ===')
    fla, fls = fl['average'], fl['std']
    for k in ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']:
        print(f'  {k:12s}: {fla[k]:.4f}  ±{fls[k]:.4f}')
    print('\n  Per-run fileless:')
    for r in report['runs']:
        if r.get('fileless'):
            m  = r['fileless']
            cm = m['confusion_matrix']
            print(f"    Run {r['run']}: acc={m['accuracy']:.4f}  rec={m['recall']:.4f}  "
                  f"f1={m['f1']:.4f}  CM: TN={cm[0][0]} FP={cm[0][1]} | FN={cm[1][0]} TP={cm[1][1]}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

keys   = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
labels = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
runs   = report['runs']
avg    = report['average']
std    = report['std']
colors = ['#4C72B0', '#DD8452', '#55A868']

# ── 1. Training curves (loss + F1 per epoch) ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i, r in enumerate(runs):
    hist   = r['history']
    epochs = [h['epoch'] for h in hist]
    axes[0].plot(epochs, [h['train_loss'] for h in hist], color=colors[i], label=f"Run {r['run']}")
    axes[1].plot(epochs, [h['f1']         for h in hist], color=colors[i], label=f"Run {r['run']}")
for ax, title, ylabel in zip(axes, ['Train Loss', 'Val F1 (macro)'], ['Loss', 'F1']):
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(); ax.grid(alpha=0.3)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/output/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 2. Per-run metrics bar + 3-run average ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x, w = np.arange(len(keys)), 0.25
ax = axes[0]
for i, r in enumerate(runs):
    ax.bar(x + i*w, [r['test'][k] for k in keys], w, label=f"Run {r['run']}", color=colors[i])
ax.set_xticks(x + w); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score'); ax.set_title('Per-run Test Metrics'); ax.legend()
for s in ['top', 'right']: ax.spines[s].set_visible(False)

ax = axes[1]
avgs = [avg[k] for k in keys]
stds = [std[k] for k in keys]
bars = ax.bar(labels, avgs, color='#4C72B0', alpha=0.85,
              yerr=stds, capsize=6, error_kw={'elinewidth': 2, 'ecolor': '#333'})
for bar, v, s in zip(bars, avgs, stds):
    ax.text(bar.get_x() + bar.get_width()/2, v + s + 0.015,
            f'{v:.3f}', ha='center', fontsize=9)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score'); ax.set_title('3-Run Average ± Std')
for s in ['top', 'right']: ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/output/metrics_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 3. Confusion matrices: all runs + fileless ─────────────────────────
fl     = report.get('fileless_generalisation') or {}
has_fl = fl.get('average') and any(r.get('fileless') for r in runs)
n_cols = len(runs) + (1 if has_fl else 0)
fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 4))
if n_cols == 1: axes = [axes]

for i, r in enumerate(runs):
    cm = np.array(r['test']['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['benign', 'webshell'],
                yticklabels=['benign', 'webshell'],
                ax=axes[i], cbar=False)
    axes[i].set_title(f"Run {r['run']}  F1={r['test']['f1']:.4f}")
    axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('True')

if has_fl:
    fl_cms = [np.array(r['fileless']['confusion_matrix']) for r in runs if r.get('fileless')]
    avg_fl_cm = np.round(np.mean(fl_cms, axis=0)).astype(int)
    fla = fl['average']
    sns.heatmap(avg_fl_cm, annot=True, fmt='d', cmap='Oranges',
                xticklabels=['benign', 'webshell'],
                yticklabels=['benign', 'webshell'],
                ax=axes[-1], cbar=False)
    axes[-1].set_title(f"Fileless held-out (avg)\nRec={fla['recall']:.4f}  F1={fla['f1']:.4f}")
    axes[-1].set_xlabel('Predicted'); axes[-1].set_ylabel('True')

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/output/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png, metrics_bar.png, confusion_matrices.png')

## Step 7 — Save results back to Google Drive

In [ ]:
import shutil, os, json

SAVE_DIR = f'{DRIVE_BASE}/results'
os.makedirs(SAVE_DIR, exist_ok=True)

files_to_save = [
    'model_best.pt',           # ResNet50 weights — copy to output/ locally
    'training_report.json',    # metrics + history for all 3 runs
    'training_curves.png',     # loss + F1 per epoch
    'metrics_bar.png',         # per-run + average bar chart
    'confusion_matrices.png',  # CMs for all runs + fileless held-out
]

for fname in files_to_save:
    src = f'{WORK_DIR}/output/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{SAVE_DIR}/{fname}')
        size_kb = os.path.getsize(src) // 1024
        print(f'  ✓ {fname}  ({size_kb} KB)')
    else:
        print(f'  ✗ {fname}  not found')

print(f'\nAll results saved to: {SAVE_DIR}')
print('\n── Next steps (local machine) ────────────────────────────────')
print('  1. Download model_best.pt      → ShellBreaker/output/')
print('  2. Download training_report.json → ShellBreaker/output/')
print('  3. Inference auto-detects ResNet50:')
print('       .venv/bin/python scripts/05_inference_api.py path/to/Suspicious.class')